In [2]:

!pip -q install -U kagglehub ultralytics scikit-learn

import os, shutil, random
from pathlib import Path

import numpy as np
import kagglehub
from ultralytics import YOLO

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


EXCLUDE_ALIASES = {
    "actinickeratosis", "actinic_keratosis", "actinic-keratosis", "actinic keratosis", "akiec",
    "seborrheickeratosis", "seborrheic_keratosis", "seborrheic-keratosis", "seborrheic keratosis", "sk"
}

def _norm(s: str) -> str:
    return "".join(ch.lower() for ch in s if ch.isalnum())

def is_excluded(class_name: str) -> bool:
    n = _norm(class_name)
    if n in {_norm(x) for x in EXCLUDE_ALIASES}:
        return True
    cl = class_name.lower()
    if ("actinic" in cl and "keratosis" in cl):
        return True
    if ("seborrheic" in cl and "keratosis" in cl):
        return True
    return False

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 123.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:

path = kagglehub.dataset_download("nodoubttome/skin-cancer9-classesisic")
print("Path to dataset files:", path)

BASE = Path(path)

def has_images(folder: Path) -> bool:

    for ext in IMG_EXTS:
        if any(folder.glob(f"*{ext}")):
            return True
    return False

def find_split_root(base: Path):
    '''
    Look for a directory containing train/val(/test) subfolders.
    Returns the split root if found, else None.
    '''
    best = None
    best_score = -1

    for p in base.rglob("*"):
        if not p.is_dir():
            continue
        train_dir = p / "train"
        val_dir   = p / "val"
        if not (train_dir.is_dir() and val_dir.is_dir()):
            continue

        class_dirs = [d for d in train_dir.iterdir() if d.is_dir() and has_images(d)]
        if len(class_dirs) < 2:
            continue

        score = sum(1 for d in class_dirs for f in d.iterdir() if f.suffix.lower() in IMG_EXTS)
        if score > best_score:
            best, best_score = p, score

    return best

def find_class_root_unsplit(base: Path) -> Path:
    '''
    Find a folder that directly contains multiple class subfolders with images.
    Used when train/val split does not exist.
    '''
    candidates = []
    for root, dirs, _ in os.walk(base):
        dirs[:] = [d for d in dirs if not d.startswith(".")]
        if len(dirs) < 2:
            continue

        root_p = Path(root)
        class_like = []
        for d in dirs:
            p = root_p / d
            if p.is_dir() and has_images(p):
                class_like.append(p)

        if len(class_like) >= 2:
            img_count = 0
            for p in class_like:
                img_count += sum(1 for f in p.iterdir() if f.suffix.lower() in IMG_EXTS)
            candidates.append((img_count, root_p))

    if not candidates:
        raise FileNotFoundError(
            "Couldn't find images. Inspect `path` printed above to locate the image folders."
        )

    candidates.sort(reverse=True, key=lambda x: x[0])
    return candidates[0][1]

split_root = find_split_root(BASE)
if split_root:
    print("Detected split dataset root:", split_root)
    src_train = split_root / "train"
    src_val   = split_root / "val"
    src_test  = split_root / "test" if (split_root / "test").is_dir() else None

    all_classes = sorted([p.name for p in src_train.iterdir() if p.is_dir() and has_images(p)])
else:
    src_train = src_val = src_test = None
    class_root = find_class_root_unsplit(BASE)
    print("Detected unsplit class_root:", class_root)
    all_classes = sorted([p.name for p in class_root.iterdir() if p.is_dir() and has_images(p)])

print("\nAll detected classes:", all_classes)

classes = [c for c in all_classes if not is_excluded(c)]
excluded = [c for c in all_classes if is_excluded(c)]

print("\nExcluded classes:", excluded)
print("Remaining classes (used):", classes)
print("Num classes:", len(classes))

assert len(classes) >= 2, "After excluding classes, you must still have at least 2 classes to train."

Using Colab cache for faster access to the 'skin-cancer9-classesisic' dataset.
Path to dataset files: /kaggle/input/skin-cancer9-classesisic
Detected unsplit class_root: /kaggle/input/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration/Train

All detected classes: ['actinic keratosis', 'basal cell carcinoma', 'dermatofibroma', 'melanoma', 'nevus', 'pigmented benign keratosis', 'seborrheic keratosis', 'squamous cell carcinoma', 'vascular lesion']

Excluded classes: ['actinic keratosis', 'seborrheic keratosis']
Remaining classes (used): ['basal cell carcinoma', 'dermatofibroma', 'melanoma', 'nevus', 'pigmented benign keratosis', 'squamous cell carcinoma', 'vascular lesion']
Num classes: 7


In [4]:

OUT_ROOT = Path("skin_isic_no_ak_sk_yolo_cls")

TRAIN_P, VAL_P, TEST_P = 0.80, 0.10, 0.10

if OUT_ROOT.exists():
    shutil.rmtree(OUT_ROOT)

for split in ["train", "val", "test"]:
    for c in classes:
        (OUT_ROOT / split / c).mkdir(parents=True, exist_ok=True)

def list_images(folder: Path):
    imgs = []
    for ext in IMG_EXTS:
        imgs.extend(folder.rglob(f"*{ext}"))
    return [p for p in imgs if p.is_file()]

counts = {c: {"train": 0, "val": 0, "test": 0, "total": 0} for c in classes}

def copy_split(src_split_dir: Path, split_name: str):
    for c in classes:
        src_c = src_split_dir / c
        if not src_c.is_dir():
            continue
        imgs = list_images(src_c)
        counts[c]["total"] += len(imgs)
        for p in imgs:
            dst = OUT_ROOT / split_name / c / p.name
            shutil.copy2(p, dst)
        counts[c][split_name] += len(imgs)

if src_train is not None and src_val is not None:

    copy_split(src_train, "train")
    copy_split(src_val, "val")
    if src_test is not None:
        copy_split(src_test, "test")
    else:

        print("No 'test' split found. Creating test split by sampling from output train split...")
        for c in classes:
            train_imgs = list((OUT_ROOT / "train" / c).glob("*"))
            random.shuffle(train_imgs)
            n = len(train_imgs)
            if n < 10:
                continue
            n_test = max(1, int(TEST_P * n))
            for p in train_imgs[:n_test]:
                dst = OUT_ROOT / "test" / c / p.name
                shutil.move(str(p), str(dst))
                counts[c]["train"] -= 1
                counts[c]["test"]  += 1
else:

    class_root = find_class_root_unsplit(BASE)
    for c in classes:
        src = class_root / c
        imgs = list_images(src)
        random.shuffle(imgs)

        n = len(imgs)
        counts[c]["total"] = n

        if n < 5:
            train_imgs, val_imgs, test_imgs = imgs, [], []
        else:
            n_test = max(1, int(TEST_P * n))
            n_val  = max(1, int(VAL_P  * n))
            n_train = max(1, n - n_val - n_test)

            train_imgs = imgs[:n_train]
            val_imgs   = imgs[n_train:n_train+n_val]
            test_imgs  = imgs[n_train+n_val:n_train+n_val+n_test]

        for split_name, split_imgs in [("train", train_imgs), ("val", val_imgs), ("test", test_imgs)]:
            for p in split_imgs:
                dst = OUT_ROOT / split_name / c / p.name
                shutil.copy2(p, dst)
            counts[c][split_name] = len(split_imgs)

print("Created:", OUT_ROOT.resolve(), "\n")
print("Per-class counts (train/val/test/total):")
for c in classes:
    t, v, te, tot = counts[c]["train"], counts[c]["val"], counts[c]["test"], counts[c]["total"]
    print(f"- {c:25s} | {t:5d} / {v:5d} / {te:5d} / {tot:5d}")


for ex in excluded:
    assert not (OUT_ROOT / "train" / ex).exists(), f"Excluded class folder still present: {ex}"

Created: /content/skin_isic_no_ak_sk_yolo_cls 

Per-class counts (train/val/test/total):
- basal cell carcinoma      |   302 /    37 /    37 /   376
- dermatofibroma            |    77 /     9 /     9 /    95
- melanoma                  |   352 /    43 /    43 /   438
- nevus                     |   287 /    35 /    35 /   357
- pigmented benign keratosis |   370 /    46 /    46 /   462
- squamous cell carcinoma   |   145 /    18 /    18 /   181
- vascular lesion           |   113 /    13 /    13 /   139


In [5]:
model = YOLO("yolov8s-cls.pt")

EPOCHS = 60
IMGSZ  = 224
BATCH  = 64


train_results = model.train(
    data=str(OUT_ROOT),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device='cpu',
    seed=SEED,
    patience=8,
)

print("Training done.")
print("Run dir:", model.trainer.save_dir if hasattr(model, "trainer") else "runs/classify/")

Ultralytics 8.4.39 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=skin_isic_no_ak_sk_yolo_cls, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=

In [6]:

run_dir = Path(model.trainer.save_dir) if hasattr(model, "trainer") else Path("runs/classify/train")
best_w = run_dir / "weights" / "best.pt"
assert best_w.exists(), f"Could not find best.pt at: {best_w}"

best_model = YOLO(str
(best_w))

test_paths = []
test_true  = []
for idx, c in enumerate(classes):
    imgs = []
    for ext in IMG_EXTS:
        imgs += list((OUT_ROOT / "test" / c).glob(f"*{ext}"))
    imgs = sorted(imgs)
    test_paths += [str(p) for p in imgs]
    test_true  += [idx] * len(imgs)

test_true = np.array(test_true, dtype=int)
print("Num test images:", len(test_paths))

preds = []
BS = 64
for i in range(0, len(test_paths), BS):
    batch = test_paths[i:i+BS]
    out = best_model.predict(batch, imgsz=IMGSZ, device='cpu', verbose=False)
    for r in out:
        preds.append(int(r.probs.top1))

preds = np.array(preds, dtype=int)

acc = accuracy_score(test_true, preds)
print("\n==================== TEST RESULTS ====================")
print("Test Accuracy:", round(acc, 4))

print("\nConfusion matrix:")
cm = confusion_matrix(test_true, preds)
print(cm)

print("\nClassification report:")
print(classification_report(
    test_true, preds,
    target_names=classes,
    digits=4,
    zero_division=0
))

Num test images: 201

==================== TEST RESULTS ====================
Test Accuracy: 0.7811

Confusion matrix:
[[31  0  1  0  3  2  0]
 [ 1  8  0  0  0  0  0]
 [ 0  0 37  5  1  0  0]
 [ 2  0  6 24  3  0  0]
 [ 2  1  2  3 37  1  0]
 [ 0  0  1  3  6  8  0]
 [ 0  0  0  0  0  1 12]]

Classification report:
                            precision    recall  f1-score   support

      basal cell carcinoma     0.8611    0.8378    0.8493        37
            dermatofibroma     0.8889    0.8889    0.8889         9
                  melanoma     0.7872    0.8605    0.8222        43
                     nevus     0.6857    0.6857    0.6857        35
pigmented benign keratosis     0.7400    0.8043    0.7708        46
   squamous cell carcinoma     0.6667    0.4444    0.5333        18
           vascular lesion     1.0000    0.9231    0.9600        13

                  accuracy                         0.7811       201
                 macro avg     0.8042    0.7778    0.7872       201
       

In [7]:
model.save("skin_model.h5")